# Chia Train / Validation / Test cho MG95 t+1, t+3, t+7

Notebook này sẽ chia dữ liệu cho **cả 3 bài toán dự báo**:

- `MG95_t_plus_1`
- `MG95_t_plus_3`
- `MG95_t_plus_7`

Tỷ lệ chia: **70% train, 15% validation, 15% test**.  
Dữ liệu được chia theo thứ tự thời gian, **không shuffle**.


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# =========================
# CẤU HÌNH ĐƠN GIẢN
# =========================

import os
import glob
import pandas as pd

# Chạy cả 3 horizon
HORIZONS = [1, 3, 7]

# Thư mục gốc dự án
SEARCH_ROOT = "/content/drive/MyDrive/DACS_2"

# Thư mục lưu kết quả split
OUTPUT_ROOT = "/content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA XỬ LÝ/Data_Train_Val_Test"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Sẽ chia dữ liệu cho:", HORIZONS)
print("Thư mục lưu kết quả:", OUTPUT_ROOT)

Sẽ chia dữ liệu cho: [1, 3, 7]
Thư mục lưu kết quả: /content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA XỬ LÝ/Data_Train_Val_Test


In [14]:
# =========================
# HÀM TÌM FILE SAU MERGE
# =========================

def find_model_file(horizon):
    file_name = f"model_dataset_MG95_t_+_{horizon}.csv"

    matched_files = glob.glob(
        os.path.join(SEARCH_ROOT, "**", file_name),
        recursive=True
    )

    if len(matched_files) == 0:
        raise FileNotFoundError(
            f"Không tìm thấy file {file_name}. "
            "Bạn cần chạy file Merge trước để tạo đủ model_dataset_MG95_t_plus_1/3/7.csv."
        )

    # Nếu có nhiều file trùng tên, lấy file mới nhất
    matched_files = sorted(matched_files, key=os.path.getmtime, reverse=True)
    return matched_files[0]

In [18]:
# =========================
# HÀM CHIA TRAIN / VAL / TEST
# =========================

def split_one_horizon(horizon):
    target_col = f"MG95_t_+_{horizon}"
    input_file = find_model_file(horizon)

    print("=" * 80)
    print(f"DỰ BÁO T+{horizon}")
    print("File đầu vào:", input_file)
    print("Target:", target_col)

    df = pd.read_csv(input_file)

    if target_col not in df.columns:
        raise KeyError(
            f"Không thấy cột target {target_col} trong file. "
            f"Các cột hiện có là: {list(df.columns)}"
        )

    # Nếu có Date thì sort theo Date
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"])
        df = df.sort_values("Date").reset_index(drop=True)
    else:
        # Nếu file không còn Date, giữ nguyên thứ tự dòng sau merge
        df = df.reset_index(drop=True)
        print("Cảnh báo: File không có cột Date, sẽ chia theo thứ tự dòng hiện tại.")

    # Bỏ dòng thiếu target nếu còn
    df = df.dropna(subset=[target_col]).reset_index(drop=True)

    n = len(df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    train_df = df.iloc[:train_end].copy()
    val_df = df.iloc[train_end:val_end].copy()
    test_df = df.iloc[val_end:].copy()

    # Tách X/y để lưu riêng
    feature_cols = [col for col in df.columns if col not in ["Date", target_col]]

    X_train = train_df[feature_cols].copy()
    y_train = train_df[[target_col]].copy()

    X_val = val_df[feature_cols].copy()
    y_val = val_df[[target_col]].copy()

    X_test = test_df[feature_cols].copy()
    y_test = test_df[[target_col]].copy()

    # Thư mục lưu riêng cho từng horizon
    output_dir = os.path.join(OUTPUT_ROOT, f"MG95_t_plus_{horizon}")
    os.makedirs(output_dir, exist_ok=True)

    # Lưu raw
    train_df.to_csv(os.path.join(output_dir, "train.csv"), index=False, encoding="utf-8-sig")
    val_df.to_csv(os.path.join(output_dir, "val.csv"), index=False, encoding="utf-8-sig")
    test_df.to_csv(os.path.join(output_dir, "test.csv"), index=False, encoding="utf-8-sig")

    # Lưu X/y riêng
    X_train.to_csv(os.path.join(output_dir, "X_train.csv"), index=False, encoding="utf-8-sig")
    y_train.to_csv(os.path.join(output_dir, "y_train.csv"), index=False, encoding="utf-8-sig")

    X_val.to_csv(os.path.join(output_dir, "X_val.csv"), index=False, encoding="utf-8-sig")
    y_val.to_csv(os.path.join(output_dir, "y_val.csv"), index=False, encoding="utf-8-sig")

    X_test.to_csv(os.path.join(output_dir, "X_test.csv"), index=False, encoding="utf-8-sig")
    y_test.to_csv(os.path.join(output_dir, "y_test.csv"), index=False, encoding="utf-8-sig")

    print("Kích thước full:", df.shape)
    print("Train:", train_df.shape)
    print("Val  :", val_df.shape)
    print("Test :", test_df.shape)

    if "Date" in df.columns:
        print("Train:", train_df["Date"].min(), "→", train_df["Date"].max())
        print("Val  :", val_df["Date"].min(), "→", val_df["Date"].max())
        print("Test :", test_df["Date"].min(), "→", test_df["Date"].max())

    print("Số feature:", len(feature_cols))
    print("Đã lưu vào:", output_dir)

    return {
        "horizon": horizon,
        "input_file": input_file,
        "output_dir": output_dir,
        "n_rows": n,
        "n_features": len(feature_cols),
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df)
    }

## CHẠY CHIA DATA CHO CẢ T+1, T+3, T+7

In [20]:
results = []

for h in HORIZONS:
    result = split_one_horizon(h)
    results.append(result)

summary_df = pd.DataFrame(results)
display(summary_df)

DỰ BÁO T+1
File đầu vào: /content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA XỬ LÝ/Merge 3 file (price_gpr_day)/model_dataset_MG95_t_+_1.csv
Target: MG95_t_+_1
Cảnh báo: File không có cột Date, sẽ chia theo thứ tự dòng hiện tại.
Kích thước full: (3998, 23)
Train: (2798, 23)
Val  : (600, 23)
Test : (600, 23)
Số feature: 22
Đã lưu vào: /content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA XỬ LÝ/Data_Train_Val_Test/MG95_t_plus_1
DỰ BÁO T+3
File đầu vào: /content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA XỬ LÝ/Merge 3 file (price_gpr_day)/model_dataset_MG95_t_+_3.csv
Target: MG95_t_+_3
Cảnh báo: File không có cột Date, sẽ chia theo thứ tự dòng hiện tại.
Kích thước full: (3996, 23)
Train: (2797, 23)
Val  : (599, 23)
Test : (600, 23)
Số feature: 22
Đã lưu vào: /content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA XỬ LÝ/Data_Train_Val_Test/MG95_t_plus_3
DỰ BÁO T+7
File đầu vào: /content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA XỬ LÝ/Merge 3 file (price_gpr_day)/model_dataset_MG95_t_+_7.csv
Target: MG95_t_+

,horizon,input_file,output_dir,n_rows,n_features,train_rows,val_rows,test_rows
0,1,/content/drive/MyDrive/DACS_2/data/DATA ĐÃ QU...,/content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA...,3998,22,2798,600,600
1,3,/content/drive/MyDrive/DACS_2/data/DATA ĐÃ QU...,/content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA...,3996,22,2797,599,600
2,7,/content/drive/MyDrive/DACS_2/data/DATA ĐÃ QU...,/content/drive/MyDrive/DACS_2/data/DATA ĐÃ QUA...,3992,22,2794,599,599
